# TP 2 - Assistant RAG simple


---
## 0. Configuration partagée


Version RAG simple : transformer la question en vecteur, récupérer les passages les plus proches, puis répondre avec ces sources. Cette version sert de référence avant les méthodes avancées de `2_4_rag_assistant_improved.ipynb`.

**TODO — `RAGAssistant`**

Fichier à modifier : `shared/rag_utils.py`

Le notebook appelle plusieurs fois la même logique de recherche. Cette classe sert à centraliser le chargement de la base Chroma et la récupération des chunks

`RAGAssistant` : classe qui charge la base Chroma et retourne des paires `(RAGChunk, score)` via la méthode `search`

Les détails des paramètres d'entrée et de sortie sont décrits dans les docstrings de `shared/rag_utils.py`

In [1]:
from shared.config import ROOT_DIR
from shared.llm_utils import LLMRequest, run_llm
from shared.rag_utils import RAGAssistant

DATA_DIR = ROOT_DIR / "TP2_travel_planner_RAG" / "data"
VECTOR_DB_DIR = DATA_DIR / "chroma_db_rag_v1"  # choisir entre chroma_db_rag_v1 et chroma_db_rag_v2 selon la base à tester

# TODO : initialiser la classe RAGAssistant
rag_assistant = RAGAssistant(
    persist_dir=VECTOR_DB_DIR)

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


---
## 1. Entrée


La requête utilisateur et le prompt système sont déjà fournis.

Concrètement, il faut vérifier avant exécution :
- que le prompt interdit toute invention,
- que chaque affirmation factuelle doit être sourcée,
- que les informations manquantes sont explicitement signalées.


In [2]:
user_query = """
Je veux partir 4 jours à Rome en avril, je n'ai pas encore les dates exactes.
Propose-moi un itinéraire. Mon budget est de 200 euros pour les sorties et les restaurants.
Je veux éviter les zones trop touristiques et découvrir des lieux plus confidentiels.
"""

system_prompt = """
Tu es un assistant de planification de voyage factuel basé sur la méthode RAG.

### Règles:
- Utiliser uniquement les faits présents dans CONTEXTE.
- Ne jamais inventer prix, dates, horaires, adresses ou transports.
- Si une information manque, écrire: "Je ne sais pas à partir du contexte fourni."
- Citer chaque affirmation factuelle au format [source - chunk id].

### Contraintes de style:
- Être concis mais précis.
- Pas de mise en forme Markdown décorative.
- Pour chaque recommandation: 1 raison courte + 1 détail pratique (horaire, lieu, budget, logistique).
- Préférer des suggestions concrètes aux phrases vagues.

### Format de réponse:
1) Résumé: réponse courte (2 à 4 phrases)
2) Plan: séquence pratique adaptée à la demande
3) Informations manquantes: liste concise des faits absents
"""

---
## 2. Récupérer le contexte


Inspecter les chunks récupérés avant génération.

Contrôles concrets:
- Pertinence: les chunks répondent-ils réellement à la requête ?
- Couverture: proviennent-ils de plusieurs sources utiles ?
- Si c'est mauvais ici, corriger le chunking/indexation, pas le prompt de génération.


In [3]:
top_chunks = rag_assistant.search(
    query=user_query,
    top_k=5,
)

INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
INFO:shared.rag_utils:Embeddings calculés avec succès pour 1 textes


**TODO** : Afficher combien de chunks ont été récupérés par source

In [5]:
print(f"Chunks récupérés : {len(top_chunks)}")
print(f"Fichiers et volume :")
source_counts = {}
for chunk, _score in top_chunks:
    source = chunk.source
    if source in source_counts:
        source_counts[source] += 1
    else:
        source_counts[source] = 1
for source, count in source_counts.items():
    print(f"  {source}: {count} chunks")
print("\n"*5)

for rank, (chunk, score) in enumerate(top_chunks, start=1):
    preview = chunk.text.replace("\n", " ")
    print(f"Chunk #{rank} | score_distance_vecteur={score:.4f} | source={chunk.source} | chunk_id={chunk.chunk_id}")
    print(preview)
    print("\n\n")

Chunks récupérés : 5
Fichiers et volume :
  rome_5_days_guide.md: 4 chunks
  rome_guide_lieux.md: 1 chunks






Chunk #1 | score_distance_vecteur=0.6336 | source=rome_5_days_guide.md | chunk_id=0
# 5-day Rome City Guide  A preplanned step-by-step time line and city guide for Rome. Follow it and get the best of the city.  ## Jour 1  ### Programme  ||**LEAVE HOTEL**|**LEAVE HOTEL**|Tested|and recommended hotels in Rome >|and recommended hotels in Rome >|| |---|---|---|---|---|---|---| ||Take Metro line B to Colosseo station|||||| |09:00-10:30|**Colosseum**||||Iconic symbol of|Page 5| ||||||Imperial Rome|| ||Take a walk to Arch of Constantine - 5'|||||| |10:35-10:45|**Arch of Constantine**||||Majestic monument|Page 5| ||Take a walk to Roman Forum|||- 5'||| |10:50-13:20|**Roman Forum and Palatine Hill**||||Center of the ancient|Page 6| ||||||world|| ||Lunch time|||||| ||Take a walk to Piazza Venezia|||||| |15:30-15:50|**Piazza Venezia**||||Focal point of modern|Page 6| ||||||Rome|| |15:50

---
## 3. Générer la réponse


Les chunks sont concaténés dans un bloc CONTEXTE, injectés dans le prompt système, puis envoyés au modèle.

Validation concrète de la sortie:
- chaque fait important doit être sourcé,
- aucune invention,
- les zones d'incertitude doivent être annoncées explicitement.


In [9]:
top_chunks = rag_assistant.search(query=user_query, top_k=5)
context = "\n".join([chunk.text for chunk, _ in top_chunks])
grounded_system_prompt = f"{system_prompt}\n\nCONTEXTE:\n{context}"
run_result = await run_llm(LLMRequest(system_prompt=grounded_system_prompt, user_prompt=user_query))

print(run_result.output)
print()
print(
    f"tokens :\n entrée={run_result.input_tokens} | sortie={run_result.output_tokens} | total={run_result.total_tokens}"
)


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-2-preview:batchEmbedContents "HTTP/1.1 200 OK"
INFO:shared.rag_utils:Embeddings calculés avec succès pour 1 textes
INFO:google_genai.models:AFC is enabled with max remote calls: 10.


Je ne peux pas proposer un itinéraire évitant les zones trop touristiques et découvrant des lieux confidentiels, car le contexte fourni se concentre principalement sur les sites touristiques majeurs de Rome. Le budget de 200 euros pour 4 jours (soit 50 euros par jour) pour les sorties et les restaurants est réalisable avec les informations disponibles sur les entrées de certains sites, mais les coûts des restaurants ne sont pas précisés.

Voici une proposition d'itinéraire basée sur les informations disponibles, en essayant de regrouper les visites par zone pour optimiser les déplacements, mais cela inclura des sites très connus.

1) Résumé:
Ce guide propose un itinéraire de 4 jours à Rome, axé sur les sites emblématiques et quelques musées. Les jours incluent des visites comme le Colisée, le Forum Romain, le Panthéon, la Fontaine de Trevi, la Villa d'Este à Tivoli, et le Musée National Étrusque de Villa Giulia. Les informations sur les coûts des repas et les lieux plus confidentiels n